# Example 01: full pipeline with a mock PCS

Read the stored campaign preshot **RTGSFIT 99000230 / PRESHOT04**, compile the C library,
feed it recorded ST40 inputs, and process its saved outputs. No preshot is built by GSFit or written to MDSplus.

```text
PRESHOT04 -> C-only build -> librtgsfit.so
ST40 TEPCS inputs -> C calls -> buffered outputs -> NetCDF capture
NetCDF capture -> offline diagnostics -> local NPZ -> optional private MDSplus run
```

Python stands in for the Plasma Control System (PCS): it supplies measurements and coil currents,
retains the iteration state, and buffers each C result. NetCDF writing and all derived diagnostics
happen **after** the loop. This demonstrates the v2 interface, not the actual PCS/Simulink integration
or real-time performance. The example samples every 1 ms; the PCS cycle is 50 microseconds.

| Step | What runs |
|---|---|
| 1 | Select pulse 14926, PRESHOT04 and local output paths |
| 2 | Read the preshot and inspect its grid and coil responses |
| 3 | Fetch the same preshot with the C generator and compile; check its checksum |
| 4 | Read calibrated recorded PCS inputs |
| 5 | Call the C solver once per sample and buffer every output |
| 6 | Write the native NetCDF capture with the shared C writer |
| 7 | Read that file, compute diagnostics and save/read back local results |
| 8 | Plot the processed results |
| 9 | Optionally write/read back a private MDSplus results run |

**Default: no database writes.** `WRITE_MDSPLUS = True` enables only the final results write,
with an unused private destination. The shared preshot and `BEST` are never changed.
See [README.md](README.md) for requirements and [helpers.py](helpers.py) for checks and plots.
Restart the kernel before running all cells; do not run another build against the same C directory.

In [ ]:
import os

# Set this before importing NumPy or loading the solver.
os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [ ]:
import subprocess
from datetime import UTC
from datetime import datetime

import helpers
import numpy as np
from rtgsfit import data_dictionary as dd
from rtgsfit import load_settings
from rtgsfit.capture import Capture
from rtgsfit.capture import read_capture
from rtgsfit.capture import write_capture
from rtgsfit.database_readers.local import read_results
from rtgsfit.database_readers.rtgsfit_mdsplus import read_mdsplus_nodes
from rtgsfit.library import Rtgsfit
from rtgsfit.library import c_directory
from rtgsfit.library import cache_directory
from rtgsfit.replay import preshot_location
from rtgsfit.replay import read_preshot
from rtgsfit.replay import read_signals
from rtgsfit.replay import write_results
from rtgsfit.results import Results

## 1. Pulse, preshot and outputs

`st40_tepcs` uses recorded PCS digitiser counts and the pulse's calibration settings, rather than
the processed magnetics used by GSFit. PRESHOT04 supplies the fixed machine/solver data; the pulse
being replayed is a separate choice. PRESHOT04 is for ST40 pulses **12551 onward**.

Every run gets a new local output directory. MDSplus writing is disabled unless explicitly enabled
below; when enabled, use your own allocated private pulse space and an unused run name.

In [ ]:
ST40_PULSE = 14_926
PRESHOT_PULSE = 99_000_230
PRESHOT_RUN = "PRESHOT04"
WRITE_MDSPLUS = False
RESULTS_PULSE = 52_000_000 + ST40_PULSE
RESULTS_RUN = "PCS_MOCK01"

if ST40_PULSE < 12_551:
    raise ValueError("This PRESHOT04 example requires ST40 pulse >= 12551")
if WRITE_MDSPLUS and (RESULTS_PULSE == PRESHOT_PULSE or RESULTS_RUN.upper() == "BEST"):
    raise ValueError("Choose a private results pulse and a run other than BEST")

stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = cache_directory().resolve() / "example_01_pcs_mock" / stamp
output_dir.mkdir(parents=True, exist_ok=False)
capture_path = output_dir / f"st40_{ST40_PULSE}.nc"
results_path = output_dir / f"st40_{ST40_PULSE}_processed.npz"

settings = load_settings("st40_tepcs").copy_with(
    {
        "replay/pulse": ST40_PULSE,
        "preshot/rtgsfit_mdsplus/pulse": PRESHOT_PULSE,
        "preshot/rtgsfit_mdsplus/run_name": PRESHOT_RUN,
        "capture/path": str(capture_path),
        "writers/names": ["local"],
        "writers/local/path": str(results_path),
        "writers/rtgsfit_mdsplus/enabled": False,
        "writers/rtgsfit_mdsplus/link_best": False,
    }
)
helpers.check_environment(write_mdsplus=WRITE_MDSPLUS)
print(f"Read-only preshot: RTGSFIT {PRESHOT_PULSE} {PRESHOT_RUN}")
print(f"Local outputs: {output_dir}")
print(f"MDSplus result writes: {WRITE_MDSPLUS}")

## 2. Read the stored campaign preshot

This reads and validates PRESHOT04; it does **not** call `build_preshot` or GSFit.
The preshot contains the grid, coil/sensor mapping, Green's functions, precomputed fit and Poisson
data, and initial plasma state. [Manual: the preshot](../../documentation/manual.md#5-the-preshot).

In [ ]:
preshot = read_preshot(settings)
print(preshot.summary())
print(f"Preshot content hash: {preshot.content_hash()}")

In [ ]:
figure = helpers.plot_preshot(preshot)

In [ ]:
figure = helpers.plot_coil_greens(preshot)

## 3. Build as the PCS does: C only

The command below fetches PRESHOT04 directly with `c/tools/write_preshot.c`, generates
`preshot_sizes.h` and `preshot_data.c`, then compiles `librtgsfit.so`. The command itself needs
**no Python**; this notebook just launches it. Timing is enabled to inspect the solver sections.

`Rtgsfit` loads the resulting library without rebuilding it and checks its dimensions and preshot
checksum against the independently read Python preshot. It initialises `psi_norm` and `mask`.
Only one build may use this C directory at a time.

![PCS-mock pipeline](preshot_to_rtgsfit.svg)

In [ ]:
c_root = c_directory().resolve()
subprocess.run(
    [
        "make",
        "-C",
        str(c_root),
        f"SHOT={PRESHOT_PULSE}",
        f"RUN_NAME={PRESHOT_RUN}",
        f"TIMING={int(settings['build']['timing'])}",
        f"ARCH_FLAGS={settings['build']['arch_flags']}",
    ],
    check=True,
)
library_path = c_root / "build" / "librtgsfit.so"
solver = Rtgsfit(library_path, preshot, timing=settings["build"]["timing"])
print(f"Loaded {library_path}; compiled preshot matches PRESHOT04")

## 4. Read the recorded PCS inputs

The reader calibrates TEPCS counts using the pulse's PSS settings and applies the available
offset, drift and pickup corrections. Missing correction metadata is reported: this is a mock,
not a guarantee of exact historical PCS inputs. Samples align with recorded 50-microsecond cycles.

`meas` follows `MEAS:NAMES`; `coil_signal` follows `COIL:SIGNALS`. The preshot's `COIL:MATRIX`
maps those original channels to the physical coil currents passed to C.
[Manual: readers](../../documentation/manual.md#73-database-readers).

In [ ]:
signals = read_signals(settings, preshot)
coil_current = signals.coil_current(preshot)
print(f"{len(signals.time)} samples: {signals.time[0] * 1e3:.1f} to {signals.time[-1] * 1e3:.1f} ms")
print(f"meas: {signals.meas.shape}; coil_signal: {signals.coil_signal.shape}")
print(f"coil_current: {coil_current.shape}; {list(preshot['COIL:NAMES'])}")
print(f"Input provenance: {signals.input_quality}")

In [ ]:
figure = helpers.plot_signals(signals, preshot)

## 5. Mock the PCS loop

One C call per sample: fit currents, solve flux, find the axis/boundary, update the iteration state.
The wrapper carries `psi_norm` and `mask` between calls and returns copies of the outputs.
Buffer **all** calls, including failures; do not post-process, write files or access MDSplus in this loop.

The Python wrapper/list allocate memory, unlike the real-time C solver and a preallocated PCS logger.
This is not a real-time timing qualification. Resetting first makes rerunning this cell reproducible.

In [ ]:
solver.reset()
slices = []
for meas, currents in zip(signals.meas, coil_current, strict=True):
    slices.append(solver(meas, currents))

print(f"Buffered {len(slices)} C calls, including {sum(item.status != 0 for item in slices)} unsuccessful calls")

## 6. After the loop: write the NetCDF capture

`write_capture` calls the shared C `rtgsfit_capture_write()` in the **same compiled library**.
It writes a NetCDF-4 file with the complete preshot, original inputs, actual C outputs, settings
and checksums. `j_phi` is saved directly, not reconstructed from next-iteration state.
A finished file is published atomically before any offline processing begins.
[Manual: capture contract](../../documentation/manual.md#76-native-netcdf-captures-replay-and-pcs).

In [ ]:
capture = Capture(preshot, signals, slices, settings.json_text, preshot_location(settings))
written = write_capture(capture_path, capture, library_path=library_path)
print(f"Native capture: {written} ({written.stat().st_size / 1e6:.1f} MB)")

## 7. Read the capture and process it offline

Everything below starts from the **file**, not the in-memory outputs. `read_capture` checks the schema,
shapes, types, preshot checksums and coil mapping. `to_results` computes boundary outlines, shape,
poloidal fields and other diagnostics without calling C, compiling, or reading the source database.

This is the same processing used by `rtgsfit post-process <capture.nc>`. Nonzero solver status marks
unsuccessful slices; status zero alone is not proof of a physical plasma. Inspect current, geometry
and residuals too.

In [ ]:
loaded_capture = read_capture(capture_path)
assert len(loaded_capture.slices) == len(slices)
np.testing.assert_array_equal([item.status for item in loaded_capture.slices], [item.status for item in slices])
np.testing.assert_array_equal(
    np.stack([item.j_phi for item in loaded_capture.slices]), np.stack([item.j_phi for item in slices])
)
results = loaded_capture.to_results()
helpers.print_replay_summary(results)
write_results(results, settings)
local_results = read_results(results_path)
helpers.compare_results(results, local_results)

## 8. Plot the processed results

These plots use the results read back from the local NPZ file. Shading marks failed calls.
The equilibrium panels show current density, flux contours, the boundary, magnetic axis and x-points.
The constraint plot compares measured and reconstructed signals at 100 ms.

In [ ]:
figure = helpers.plot_time_traces(local_results)

In [ ]:
figure = helpers.plot_equilibria(local_results, times=(0.03, 0.08, 0.13, 0.2))

In [ ]:
figure = helpers.plot_constraints(local_results, time=0.1)

In [ ]:
figure = helpers.plot_timing(local_results)

## 9. Optional: store processed results in MDSplus and read them back

Disabled by default. To include this final step, choose your private `RESULTS_PULSE` and an unused
`RESULTS_RUN`, set `WRITE_MDSPLUS = True`, and rerun from the start. The matching `standard_utility`
data dictionary and database write access are required. Existing runs are never deliberately
overwritten, `BEST` is not linked, and PRESHOT04 remains read-only.

In [ ]:
if WRITE_MDSPLUS:
    if RESULTS_PULSE == PRESHOT_PULSE or RESULTS_RUN.upper() == "BEST":
        raise ValueError("Choose a private results pulse and a run other than BEST")
    if not np.any(results["GLOBAL.STATUS"] == 0):
        raise RuntimeError("No C call succeeded; inspect the local capture before publishing")
    helpers.check_run_is_free(RESULTS_PULSE, RESULTS_RUN)
    mdsplus_settings = settings.copy_with(
        {
            "writers/names": ["rtgsfit_mdsplus"],
            "writers/rtgsfit_mdsplus/enabled": True,
            "writers/rtgsfit_mdsplus/link_best": False,
            "writers/rtgsfit_mdsplus/pulse": RESULTS_PULSE,
            "writers/rtgsfit_mdsplus/run_name": RESULTS_RUN,
            "writers/rtgsfit_mdsplus/run_description": (
                f"Example 01 PCS mock: ST40 {ST40_PULSE}, preshot RTGSFIT {PRESHOT_PULSE} {PRESHOT_RUN}"
            ),
        }
    )
    write_results(local_results, mdsplus_settings)
    stored = Results(read_mdsplus_nodes(RESULTS_PULSE, RESULTS_RUN, dd.ALL_NODES))
    helpers.compare_results(local_results, stored)
else:
    print("MDSplus write/readback skipped: WRITE_MDSPLUS = False; local capture and results are complete")

## Command-line equivalents

From the repository root, the Python-free PCS build is:

```sh
make -C c SHOT=99000230 RUN_NAME=PRESHOT04 TIMING=1
```

For a shorter Python replay followed by reprocessing its saved capture:

```sh
rtgsfit replay st40_tepcs --set replay/pulse=14926 \
    --set preshot/rtgsfit_mdsplus/pulse=99000230 \
    --set preshot/rtgsfit_mdsplus/run_name=PRESHOT04 \
    --set capture/path=example_01_pcs_mock/st40_14926.nc
rtgsfit post-process c/cache/example_01_pcs_mock/st40_14926.nc
```

Unlike the explicit loop above, `replay` also post-processes in memory after writing its capture.
Both commands keep database writes disabled by default. In an installed package, relative output
paths use the user cache rather than `c/cache`; use the capture path printed by the command.

The [manual](../../documentation/manual.md) describes the algorithm, API and capture contract.
Creating or publishing a new campaign preshot, and integrating the v2 API into PCS/Simulink,
are separate tasks; this notebook does neither.